In [ ]:
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import math
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

device = "cuda:0"
d = 250
m = 1000
v = 1000
# s = torch.tensor([17,258,99,291,102,197,333,66,225,92], device = device)
s = torch.randint(0, v, (1000,), device=device)

# torch.manual_seed(4)

class W2Model(nn.Module):
    def __init__(self, d, m, v, g, device) -> None:
        super().__init__()
        self.K = torch.randn(d, m, device = device, requires_grad = True) / math.sqrt(m)
        self.bias_K = torch.randn(m, device = device, requires_grad = True) / math.sqrt(m)
        self.V = torch.randn(m, d, device = device, requires_grad = True) / math.sqrt(d)
        self.bias_V = torch.randn(d, device = device, requires_grad = True) / math.sqrt(d)
        self.W = torch.randn(d, v, device = device, requires_grad = True) / math.sqrt(d)
        self.d = d
        self.g = g
        self.device = device

    def forward(self, x, return_inter = False, custom_inter = None):
        if custom_inter is None:
            inter = self.g(x @ self.K + self.bias_K)
        else:
            inter = custom_inter
        output = inter @ self.V + self.bias_V + x
        if return_inter:
            return output, inter
        else:
            return output
    
    def forward_with_params(self, x, K, V, b1 = None, b2 = None):
        if b1 is None:
            b1 = self.bias_K
        if b2 is None:
            b2 = self.bias_V
        inter = self.g(x @ K + b1)
        return inter @ V + b2 + x

    def cross_entropy(self, x, y):
        x = x.view(-1, x.size(-1))
        y = F.one_hot(y, v).view(-1, v)
        likelihood = F.log_softmax(x, -1)
        return -torch.sum(likelihood * y) / x.size(0)

    def compute_loss(self, x, y):
        z = x @ self.W
        return self.cross_entropy(z, y)
    
    def compute_logits(self, z):
        return z @ self.W


g = lambda x: x * torch.sigmoid(x)
# dg = lambda x: (x > 0).float()

x = torch.randn(s.size(0), d, device = device) / math.sqrt(d)
y = None
# y = W[None, :, s]

lr = 1e-0
max_iteration = 500
top_num = 3
loss_history = []
max_activation_inter = []
max_activation_info = []
colors = []

model = W2Model(d, m, v, g, device)

for i in tqdm(range(max_iteration)):
    z = model.forward(x)
    z.requires_grad_(True)

    loss = model.compute_loss(z, s)
    loss_history += [loss.item()]
    
    dz, dK, dV = torch.autograd.grad(loss, [z, model.K, model.V])
    dz, dK, dV = -dz, -dK, -dV

    y = dz.detach()

    # dz = (torch.eye(v, device = device)[s] - F.softmax(z @ W, -1)) @ W.T
    # dK = x.T @ (dg(x @ K) * (dz @ V.T))
    # dV = g(x @ K).T @ dz
    with torch.no_grad():
        model.K += lr * dK
        model.V += lr * dV

    Kn = x @ model.K
    Vn = dz @ model.V.T

    mKs = torch.topk(Kn , dim = 1, k = top_num)[1].tolist()
    mVs = torch.topk(Vn , dim = 1, k = top_num)[1].tolist()

    batch_act_info = []
    batch_act_inter = []
    for mK, mV in zip(mKs, mVs):
        mKn, mVn = mK[0], mV[0]
        batch_act_inter += [set(mK) & set(mV)]
        batch_act_info += [(set(mK), set(mV))]

    max_activation_inter += [batch_act_inter]
    max_activation_info += [batch_act_info]
    
    if all([len(item) > 0 for item in batch_act_inter]):
        colors += ["green"]
        # break
    else:
        colors += ["red"]

idxs = max_activation_info[-1]


# print(max_activation_info)
plt.figure(figsize = (10, 8), dpi = 100)
plt.xlabel("iteration num", fontsize=14)
plt.ylabel("loss", fontsize=14)

#plt.scatter([], [], color="red", label="different activation")
#plt.scatter([], [], color="green", label="identical activation")

#for i in range(len(loss_history)):
#    plt.scatter(i, loss_history[i], color=colors[i], s=5, zorder=2)
sns.lineplot(x = list(range(len(loss_history))), y = loss_history, zorder=1)

# plt.legend(loc='upper right', fontsize=16)
plt.show()

In [ ]:
class RewritingModel(object):
    def __init__(self, model) -> None:
        super().__init__()
        self.model = model
        self.d, self.m = self.model.K.size()

    def complete_orthogonal_matrix(self, C):
        p, n = C.shape
        assert p >= n, "C 的行数必须 ≥ 列数"
        
        Q_rand = torch.randn(p, p - n, device=C.device)
        Q_rand -= C @ (C.T @ Q_rand)
        
        Q_orth, _ = torch.linalg.qr(Q_rand)
        
        Q = torch.cat([C, Q_orth], dim=1)
        return Q
    
    def solve_Procrustes(self, A, B):
        device = A.device
        U, _, V_T = torch.linalg.svd(B.T @ A)
        I_uv = torch.eye(V_T.size(0), U.size(0), device = device)
        Omega = V_T.T @ I_uv @ U.T
        return Omega
    
    def solve_Procrustes_multi(self, A, B, C, D):
        device = A.device
        U, _, V_T = torch.linalg.svd(B.T @ A + D.T @ C)
        I_uv = torch.eye(V_T.size(0), U.size(0), device = device)
        Omega = V_T.T @ I_uv @ U.T
        return Omega

    def rewrite_K(self, X):
        K = self.model.K
        C = self.solve_Procrustes(K, X.T)
        
        return self.complete_orthogonal_matrix(C)
    
    def rewrite_V(self, Y):
        V = self.model.V
        C = self.solve_Procrustes(V.T, Y.T)
        return self.complete_orthogonal_matrix(C)
    
    def rewrite_KV(self, X, Y):
        K = self.model.K
        V = self.model.V
        C = self.solve_Procrustes_multi(K, X.T, V.T, Y.T)
        return self.complete_orthogonal_matrix(C)
    
    def asymmetric_rewrite(self, X, Y, lambda_recon = 1e-1, lambda_reg = 1):
        K = self.model.K
        V = self.model.V

        n = K.shape[1]
        m = V.shape[0]

        X_pad = torch.zeros((n, n), device=K.device)
        Y_pad = torch.zeros((m, m), device=V.device)

        c_V = torch.norm(V, p='fro')**2
        c_K = torch.norm(K, p='fro')**2

        I_A = torch.eye(n, device=K.device)
        I_B = torch.eye(m, device=V.device)

        KTK = K.T @ K
        VTV = V @ V.T

        X_fill = X @ K
        Y_fill = V @ Y.T
        X_pad[:X_fill.shape[1], :X_fill.shape[0]] = X_fill.T
        Y_pad[:Y_fill.shape[1], :Y_fill.shape[0]] = Y_fill.T

        A = torch.linalg.inv(lambda_recon * c_V * KTK + lambda_reg * I_A) @ (lambda_recon * c_V * KTK + X_pad)
        B = torch.linalg.inv(lambda_recon * c_K * VTV + lambda_reg * I_B) @ (lambda_recon * c_K * VTV + Y_pad)

        return A, B

RM = RewritingModel(model)
print(y.size(), x.size())
C = RM.rewrite_K(x)
C = RM.rewrite_V(y)
C = RM.rewrite_KV(x, y)
print(C.T @ C)
#A, B = RM.asymmetric_rewrite(x, y)
#A_inv, B_inv = torch.linalg.pinv(A), torch.linalg.pinv(B)
#C = (A, A_inv, B, B_inv)

In [ ]:
origin = model.forward(x)
optimized = model.forward_with_params(x, model.K @ A, B @ model.V)
random = model.forward_with_params(x, model.K @ torch.randn_like(A), torch.randn_like(B) @ model.V)
Q = torch.linalg.qr(torch.randn_like(A))[0]
ortho = model.forward_with_params(
    x, 
    model.K @ Q, 
    Q.T @ model.V
)
torch.norm(optimized - origin), torch.norm(random - origin), torch.norm(ortho - origin)

In [ ]:
def ppl_delta_with_C(model, x, s, C, idx, kv = 'KV'):
    with torch.no_grad():
        xi = x[idx:idx+1]
        si = s[idx:idx+1]
        x = torch.cat([x[:idx], x[idx+1:]], 0)
        s = torch.cat([s[:idx], s[idx+1:]], 0)

        zi = model.forward(xi)
        logits_old_i = model.compute_logits(zi)
        loss_old_i = model.compute_loss(zi, si)

        z = model.forward(x)
        logits_old = model.compute_logits(z)
        loss_old = model.compute_loss(z, s)

        if isinstance(C, tuple):
            A, A_inv, B, B_inv = C
            I = (xi @ model.K + model.bias_K) @ A
            W = torch.diag(F.softmax(1e4 * I, -1).squeeze())
            M_K = torch.eye(A.size(0), device=A.device) - A @ W @ A_inv
            M_V = torch.eye(B.size(0), device=B.device) - B_inv @ W @ B

            nK = model.K @ M_K
            nb = model.bias_K @ M_K
            nV = M_V @ model.V
        else:
            # I = xi @ model.K @ C + model.bias_K @ C
            # W = torch.diag(F.softmax(1e4 * I, -1).squeeze())
            c_i = C[:,idx].unsqueeze(1)
            M = torch.eye(C.size(0), device = C.device) - c_i @ c_i.T

            if 'K' in kv:
                nK = model.K @ M
                nb = model.bias_K @ M
            else:
                nK = model.K
                nb = model.bias_K
            if 'V' in kv:
                nV = M @ model.V
            else:
                nV = model.V
        
        zi = model.forward_with_params(xi, nK, nV, nb)
        logits_new_i = model.compute_logits(zi)
        loss_new_i = model.compute_loss(zi, si)

        z = model.forward_with_params(x, nK, nV, nb)
        logits_new = model.compute_logits(z)
        loss_new = model.compute_loss(z, s)

    return (
        torch.exp(loss_old_i).item(), 
        torch.exp(loss_new_i).item(),
        torch.exp(loss_old).item(), 
        torch.exp(loss_new).item(),
    ), (
        logits_old_i,
        logits_new_i,
        logits_old,
        logits_new,
    )

case_results = []
for i in range(x.size(0)):
    outputs, output_logits = ppl_delta_with_C(model, x, s, C, i, 'V')
    case_results.append({
        'old': output_logits[0].squeeze().cpu(), 
        'new': output_logits[1].squeeze().cpu(), 
        'ans': s[i],
        'old_local': output_logits[2], 
        'new_local': output_logits[3],
        'local_ans': torch.cat([s[:i], s[i+1:]], 0),
    })
    print("To erase (before -> after): ({:.6f} -> {:.6f}), To retain (before -> after): ({:.6f} -> {:.6f})".format(*outputs))

In [ ]:
from src.evaluate import erase_acc, ppl, locality
def evaluate_logits(case_results):
    result_dict = dict()
    result_dict.update(erase_acc(case_results))
    result_dict.update(ppl(case_results))
    result_dict.update(locality(case_results))
    print(
        '\n'.join(
            ["{}: {:.4f}".format(key, value) 
            for (key, value) in result_dict.items()]
        )
    )
evaluate_logits(case_results)

In [ ]:
def locate_kn_ig(
    model,
    source_target,
):
    def scaled_input(emb, num_cuts):
        # emb: (1, ffn_size)
        baseline = torch.zeros_like(emb)  # (1, ffn_size)
        step = (emb - baseline) / num_cuts  # (1, ffn_size)
        res = torch.cat([torch.add(baseline, step * i) for i in range(num_cuts)], dim=0)  # (num_points, ffn_size)
        return res, step[0]
    
    num_cuts = 20
    ig_grads = []
    for dic in tqdm(source_target):
        x, y = dic['x'], dic['y']
        repeated_x = x.repeat([num_cuts, 1])

        _, inter = model.forward(x, return_inter=True)

        scaled_weights, weights_step = scaled_input(inter, num_cuts)
        scaled_weights.requires_grad_(True)
        
        z = model(repeated_x, custom_inter = scaled_weights)
        logits = model.compute_logits(z)
        tgt_prob = F.softmax(logits, -1)
        gradient = torch.autograd.grad(torch.unbind(tgt_prob[:, y]), scaled_weights)[0]
        ig_grad = gradient.sum(dim = 0) * weights_step
        ig_grads.append(ig_grad)
    ig_grads = torch.stack(ig_grads)

    return ig_grads

xy = [{'x':x[i].unsqueeze(0), 'y':s[i]} for i in range(s.size(0))]
ig = locate_kn_ig(model, xy)
print(ig.size())

In [ ]:
def ppl_delta_with_C(model, x, s, ig, idx):
    with torch.no_grad():
        xi = x[idx:idx+1]
        si = s[idx:idx+1]
        x = torch.cat([x[:idx], x[idx+1:]], 0)
        s = torch.cat([s[:idx], s[idx+1:]], 0)

        zi = model.forward(xi)
        logits_old_i = model.compute_logits(zi)
        loss_old_i = model.compute_loss(zi, si)

        z = model.forward(x)
        logits_old = model.compute_logits(z)
        loss_old = model.compute_loss(z, s)

        # I = xi @ model.K @ C + model.bias_K @ C
        I = ig[idx]

        kn = torch.argmax(I, -1)
        # kn = idx

        nK, nb, nV = model.K.clone(), model.bias_K.clone(), model.V.clone()
        nK[:, kn], nb[kn], nV[kn, :] = 0, 0, 0      
        zi = model.forward_with_params(xi, nK, nV, nb)
        logits_new_i = model.compute_logits(zi)
        loss_new_i = model.compute_loss(zi, si)

        z = model.forward_with_params(x, nK, nV, nb)
        logits_new = model.compute_logits(z)
        loss_new = model.compute_loss(z, s)

    return (
        torch.exp(loss_old_i).item(), 
        torch.exp(loss_new_i).item(),
        torch.exp(loss_old).item(), 
        torch.exp(loss_new).item(),
    ), (
        logits_old_i,
        logits_new_i,
        logits_old,
        logits_new,
    )

print(C.size())
case_results = []
for i in range(x.size(0)):
    outputs, output_logits = ppl_delta_with_C(model, x, s, ig, i)
    case_results.append({
        'old': output_logits[0].squeeze().cpu(), 
        'new': output_logits[1].squeeze().cpu(), 
        'ans': s[i],
        'old_local': output_logits[2], 
        'new_local': output_logits[3],
        'local_ans': torch.cat([s[:i], s[i+1:]], 0),
    })
    print("To erase (before -> after): ({:.6f} -> {:.6f}), To retain (before -> after): ({:.6f} -> {:.6f})".format(*outputs))

In [ ]:
from src.evaluate import erase_acc, ppl, locality
def evaluate_logits(case_results):
    result_dict = dict()
    result_dict.update(erase_acc(case_results))
    result_dict.update(ppl(case_results))
    result_dict.update(locality(case_results))
    print(
        '\n'.join(
            ["{}: {:.4f}".format(key, value) 
            for (key, value) in result_dict.items()]
        )
    )
evaluate_logits(case_results)

In [ ]:

class ReweightingModel(nn.Module):
    def __init__(self, model, m, device) -> None:
        super().__init__()
        self.C = nn.Parameter(torch.randn(m, m, device = device) / math.sqrt(m))
        self.model = model
        self.device = device
        self.ortho_coef = 1
        self.entro_coef = 1
        self.recon_coef = 1
    # discretization loss
    def discretization(self, x, eps = 1e-8):
        x = F.softmax(x, dim = -1)
        l = -torch.diag(x @ torch.log(x + eps).T).mean()
        return l
    # index-wise discretization loss
    def cross_entropy(self, x, y):
        v = x.size(-1)
        x = x.view(-1, v)
        y = F.one_hot(y, v).view(-1, v)
        assert x.size() == y.size()
        likelihood = F.log_softmax(x, -1)
        return -torch.sum(likelihood * y) / x.size(0)
    # reconstruction loss
    def reconstruction(self, X):
        with torch.no_grad():
            Y = self.model(X)
            K = self.model.K.detach() @ self.C
            V = self.C.T @ self.model.V.detach()
            Z = self.model.g(X @ K + self.model.bias_K) @ V + self.model.bias_V
        kl_div = F.kl_div(
            F.log_softmax(Z, -1), 
            F.softmax(Y, -1),
            reduction = "batchmean"
        )
        res = (Y - Z).square().mean()
        return res
    # orthogonal reconstruction
    def orthogonalization(self, Q):
        size = Q.size(1)
        I = torch.eye(size, device=self.device)
        R = Q.T @ Q - I
        l = R.square().mean()
        return l
    # forward
    def forward(self, X, idxs = None):
        K = self.model.K.detach() @ self.C
        I = self.model.g(X @ K + self.model.bias_K.detach())
        if idxs is not None:
            entro = self.cross_entropy(I, idxs) * self.entro_coef
        else:
            entro = torch.mean(torch.square(K.T[:X.size(0)] - X))
        ortho = self.orthogonalization(self.C) * self.ortho_coef
        recon = self.reconstruction(X) * self.recon_coef
        return entro, ortho, recon

Rmodel = ReweightingModel(model, m, device)
opt = optim.Adam(
    [
        {'params': Rmodel.C},
    ],
    lr = 2e-2,
)

entro_loss_history = []
ortho_loss_history = []
recon_loss_history = []

for i in range(400):
    idxs = torch.arange(0, x.size(0), device=device)
    entro, ortho, recon = Rmodel(x, idxs = None)
    loss = entro + ortho
    opt.zero_grad()
    loss.backward()
    opt.step()
    entro_loss_history += [entro.item()]
    ortho_loss_history += [ortho.item()]
    recon_loss_history += [recon.item()]

print("Loss: {:.4f}({:.4f}, {:.4f}, {:.4f}) -> {:.4f}({:.4f}, {:.4f}, {:.4f})".format(
    sum([entro_loss_history[0], ortho_loss_history[0], recon_loss_history[0]]),
    entro_loss_history[0], ortho_loss_history[0], recon_loss_history[0],
    sum([entro_loss_history[-1], ortho_loss_history[-1], recon_loss_history[-1]]),
    entro_loss_history[-1], ortho_loss_history[-1], recon_loss_history[-1],
))
C = Rmodel.C

plt.figure(figsize = (10, 8), dpi = 100)
sns.lineplot(x = list(range(len(entro_loss_history))), y = entro_loss_history, zorder=1, label = "entropy")
sns.lineplot(x = list(range(len(ortho_loss_history))), y = ortho_loss_history, zorder=1, label = "orthogonal")
sns.lineplot(x = list(range(len(recon_loss_history))), y = recon_loss_history, zorder=1, label = "reconstruction")
plt.legend(loc = "upper right")
plt.show()

In [ ]:
import numpy as np
import torch

def gram_schmidt_complete(Q_k):
    n, k = Q_k.size()
    Q = torch.zeros((n, n-k), device = Q_k.device)
    Q = torch.cat([Q_k, Q], dim = 1)

    for i in range(k, n):
        v = torch.rand(n, device=Q.device)
        for j in range(i):
            v -= torch.matmul(Q[:, j], v) * Q[:, j]
        v /= torch.norm(v)
        Q[:, i] = v

    return Q

# 示例：给定一个 4x2 的正交矩阵，补全为 4x4 正交矩阵
Q_k = torch.tensor([[0.6, 0.8], [0, 0], [0.8, -0.6], [0, 0]])  # 4x2 的正交矩阵
Q = gram_schmidt_complete(Q_k)
print(Q)
print("Q^T Q =\n", np.round(Q.T @ Q, 6))  # 验证正交性


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np

device = "cuda"

d_in = 768
d_out = 768
rank_X = 3072
batch_size = 362

# 训练超参数
lr = 1e-2
epochs = 10000
tol = 1e-3

torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
np.random.seed(42)

x = torch.load("./outputs/te.pt")

# 生成目标输出 T（后续让模型拟合这个T）
P = torch.randperm(batch_size, device = device)
P = torch.eye(batch_size, device=device)[P]
T = P @ x

class SingleLinearLayer(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.W = nn.Parameter(torch.randn(d_in, d_out))
    
    def forward(self, x):
        return x @ self.W + x

single_model = SingleLinearLayer(d_in, d_out).to(device)
optimizer_single = optim.SGD(single_model.parameters(), lr=lr)
loss_fn = nn.MSELoss()

print("===== 训练单矩阵线性层 =====")
for epoch in range(epochs):
    single_model.train()
    optimizer_single.zero_grad()
    
    # 前向传播
    y_pred = single_model(x)
    loss = loss_fn(y_pred, T)
    
    # 反向传播 + 优化
    loss.backward()
    optimizer_single.step()
    
    # 检查收敛
    if loss.item() < tol:
        break
    
print(f"单矩阵模型收敛，epoch={epoch + 1}, loss={loss.item():.8f}")
X = single_model.W.clone()

class DecomposedLinearLayer(nn.Module):
    def __init__(self, d_in, d_out, rank):
        super().__init__()
        self.rank = rank
        # 定义分解后的矩阵K和V（无bias）
        self.fc_in = nn.Linear(d_in, rank, bias = False)
        self.fc_out = nn.Linear(rank, d_out, bias = False)
        self.cache = dict({
            "K0": self.fc_in.weight.detach().clone(), 
            "V0": self.fc_out.weight.detach().clone()
        })
    
    def forward(self, x):
        # 前向传播：x -> xKV
        return self.fc_out(self.fc_in(x)) + x


decomp_model = DecomposedLinearLayer(d_in, d_out, rank_X).to(device)

optimizer_decomp = optim.SGD(decomp_model.parameters(), lr=lr)

# 训练分解模型
print("\n===== 训练双矩阵分解层 =====")
for epoch in range(epochs):
    decomp_model.train()
    optimizer_decomp.zero_grad()
    
    # 前向传播
    y_pred = decomp_model(x)
    loss = loss_fn(y_pred, T)
    
    # 反向传播 + 优化
    loss.backward()
    optimizer_decomp.step()
    
    # 检查收敛
    if loss.item() < tol:
        break

print(f"分解模型收敛，epoch={epoch + 1}, loss={loss.item():.8f}")
K_final = decomp_model.fc_in.weight.clone().T
V_final = decomp_model.fc_out.weight.clone().T
Y = K_final @ V_final
R = torch.linalg.pinv(x.T @ x) @ x.T @ T

print("\n===== 最终验证 =====")
error = torch.norm(X - Y).item()
single_error = torch.norm(X - R).item()
decomp_error = torch.norm(Y - R).item()
print(f"X与Y的L2范数误差: {error:.8f}")
print(f"X与R的L2范数误差: {single_error:.8f}")
print(f"Y与R的L2范数误差: {decomp_error:.8f}")

decomp_model.cache["perm"] = P
decomp_model.cache["X"] = x
save = {"state_dict": decomp_model.state_dict(), "cache": decomp_model.cache}
torch.save(save, "./outputs/kv.pt")

In [ ]:
dK = decomp_model.fc_in.weight - decomp_model.cache["K0"].cuda()
# ||dK - E X||
E = dK @ torch.linalg.pinv(x)
torch.norm(dK - E @ x)
